
# VoiceGrad 验证集 checkpoint sweep 评估（Colab）

这个 notebook 是按你之前 **closed-set test notebook** 的写法改成的 **validation sweep** 版。

它会做这几件事：

1. 对 `epochs=4000, batch_size=16, lr=1e-4` 这一条 VoiceGrad(DPM+BNF) 训练线的所有保存 checkpoint 逐个做 **closed-set validation conversion**
2. validation 按你当前项目协议使用 **1001–1100**，也就是 **每个闭集说话人 100 条**，因此总量是 **12 个方向 × 100 条 = 1200 条 / checkpoint**
3. 逐个 checkpoint 计算 **CER / MCD / LFC / pMOS(DNSMOS)**  
4. 最后自动生成 **validation summary**，并和你已经有的 **test-set summary** 放在一起，导出成对比表

默认评测对象：
- `best_model.pt`
- `model_epoch_500.pt`
- `model_epoch_1000.pt`
- `model_epoch_1500.pt`
- `model_epoch_2000.pt`
- `model_epoch_2500.pt`
- `model_epoch_3000.pt`
- `model_epoch_3500.pt`
- `model_epoch_4000.pt`

`latest_model.pt` 默认不单独跑，因为它通常和 4000 重复；你要跑的话把 `INCLUDE_LATEST = True` 即可。


In [ ]:

# ===== Cell 1: 挂载 Google Drive =====
from google.colab import drive
drive.mount('/content/drive')


In [ ]:

# ===== Cell 2: 安装依赖 =====
!pip -q install transformers jiwer soundfile librosa pyworld pysptk pandas tqdm scipy speechmos onnxruntime


In [ ]:

# ===== Cell 3: 路径与配置（先改这里） =====
import os

PROJECT_ROOT = "/content/drive/MyDrive/VoiceGrad"
DATA_ROOT = os.path.join(PROJECT_ROOT, "data")
CKPT_DIR = os.path.join(PROJECT_ROOT, "checkpoints")

# HiFi-GAN
HIFIGAN_ROOT = os.path.join(PROJECT_ROOT, "hifi-gan")
HIFIGAN_CONFIG = os.path.join(HIFIGAN_ROOT, "config_16k.json")
HIFIGAN_CKPT = os.path.join(HIFIGAN_ROOT, "g_00105000")

# transcript
TEXT_PATH = os.path.join(DATA_ROOT, "cmuarctic.data.txt")

# 输出根目录
RUN_NAME = "val_ckpt_sweep_closedset_eval"
OUT_BASE = os.path.join(PROJECT_ROOT, RUN_NAME)
os.makedirs(OUT_BASE, exist_ok=True)

# 要评测的 checkpoint
EVAL_CKPTS = ["best", 500, 1000, 1500, 2000, 2500, 3000, 3500, 4000]
INCLUDE_LATEST = False
if INCLUDE_LATEST:
    EVAL_CKPTS = EVAL_CKPTS + ["latest"]

# reverse diffusion 起点：沿用你当前最终 DPM 设定
START_LEVEL = 11

# 断点续跑设置
SKIP_IF_DONE = True
REUSE_EXISTING_WAVS = True
REUSE_EXISTING_METRICS = True

# 是否评测这些指标
DO_CER = True
DO_MCD_LFC = True
DO_PMOS = True

# 验证集索引范围：严格按你当前报告
VAL_START = 1001
VAL_END = 1100

# 设备
DEVICE = "cuda" if os.path.exists("/usr/local/cuda") else "cpu"

print("PROJECT_ROOT =", PROJECT_ROOT)
print("DATA_ROOT    =", DATA_ROOT)
print("CKPT_DIR     =", CKPT_DIR)
print("HIFIGAN_ROOT =", HIFIGAN_ROOT)
print("TEXT_PATH    =", TEXT_PATH)
print("OUT_BASE     =", OUT_BASE)
print("DEVICE       =", DEVICE)
print("EVAL_CKPTS   =", EVAL_CKPTS)


In [ ]:

# ===== Cell 4: 导入 =====
import gc
import glob
import json
import math
import re
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import soundfile as sf
import librosa
import torch
import torch.nn.functional as F
from tqdm import tqdm

import pyworld as pw
import pysptk
from scipy.spatial.distance import cdist
from librosa.sequence import dtw

from jiwer import cer
from transformers import Wav2Vec2Processor, Wav2Vec2ForCTC
from speechmos import dnsmos

if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)
os.chdir(PROJECT_ROOT)

from model import VoiceGrad
from diffusion import VoiceGradDiffusion

CLOSEDSET_SPKS = ["clb", "bdl", "slt", "rms"]
SPK2ID = {"clb": 0, "bdl": 1, "slt": 2, "rms": 3}
PAIR_ORDER = [
    ("bdl", "clb"), ("bdl", "slt"), ("bdl", "rms"),
    ("clb", "bdl"), ("clb", "slt"), ("clb", "rms"),
    ("slt", "clb"), ("slt", "bdl"), ("slt", "rms"),
    ("rms", "clb"), ("rms", "bdl"), ("rms", "slt"),
]

ASR_MODEL_NAME = "facebook/wav2vec2-large-960h-lv60-self"

SR = 16000
MCEP_ORDER = 24
MCEP_ALPHA = 0.42
FRAME_PERIOD_MS = 10.0
MCD_CONST = 10.0 / np.log(10.0) * np.sqrt(2.0)

print("Imports OK")


In [ ]:

# ===== Cell 5: 通用函数 =====
def ensure_mel_shape(mel):
    if mel.ndim != 2:
        raise ValueError(f"mel shape wrong: {mel.shape}")
    if mel.shape[0] == 80:
        return mel
    if mel.shape[1] == 80:
        return mel.T
    raise ValueError(f"mel shape wrong: {mel.shape}")

def ensure_bnf_shape(bnf):
    if bnf.ndim != 2:
        raise ValueError(f"bnf shape wrong: {bnf.shape}")
    if bnf.shape[0] == 144:
        return bnf
    if bnf.shape[1] == 144:
        return bnf.T
    raise ValueError(f"bnf shape wrong: {bnf.shape}")

def resample_bnf_to_mel_len(bnf, mel_len):
    bnf_tensor = torch.from_numpy(bnf).float().unsqueeze(0)
    bnf_tensor = F.interpolate(
        bnf_tensor,
        size=mel_len,
        mode="linear",
        align_corners=False
    )
    return bnf_tensor.squeeze(0).cpu().numpy()

def load_stats(data_root):
    mean = np.load(os.path.join(data_root, "stats", "mel_mean.npy"))
    std = np.load(os.path.join(data_root, "stats", "mel_std.npy"))
    mean = torch.from_numpy(mean).float().view(1, 80, 1)
    std = torch.from_numpy(std).float().view(1, 80, 1)
    return mean, std

def normalize_mel(mel, mean, std):
    mel = torch.from_numpy(mel).float().unsqueeze(0)
    mel = (mel - mean) / std
    return mel

def denormalize_mel(mel_norm, mean, std):
    return mel_norm * std + mean

def find_bnf_path(data_root, spk, mel_fname):
    bnf_dir = os.path.join(data_root, "bnf", spk)
    p1 = os.path.join(bnf_dir, mel_fname.replace(".npy", ".ling_feat.npy"))
    p2 = os.path.join(bnf_dir, mel_fname)
    if os.path.exists(p1):
        return p1
    if os.path.exists(p2):
        return p2
    raise FileNotFoundError(f"BNF not found for {spk}/{mel_fname}")

def load_sample_from_data(data_root, spk, mel_fname):
    mel_path = os.path.join(data_root, "mel", spk, mel_fname)
    bnf_path = find_bnf_path(data_root, spk, mel_fname)

    mel = ensure_mel_shape(np.load(mel_path))
    bnf = ensure_bnf_shape(np.load(bnf_path))
    bnf = resample_bnf_to_mel_len(bnf, mel.shape[1])

    return mel, bnf, mel_path, bnf_path

def parse_global_idx(fname):
    nums = re.findall(r'\d+', fname)
    local_idx = int(nums[-1])
    if 'arctic_b' in fname or '_b' in fname:
        return local_idx + 593
    return local_idx

def normalize_text(s):
    s = s.lower().strip()
    s = re.sub(r"[^a-z0-9\s']", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def load_ref_map(text_path):
    if text_path is None:
        raise ValueError("TEXT_PATH is None.")

    ref_map = {}

    with open(text_path, "r", encoding="utf-8", errors="ignore") as f:
        lines = f.readlines()

    for line in lines:
        line = line.strip()
        if not line:
            continue

        m = re.match(r'\(\s*(arctic_[ab]\d+)\s+"(.+)"\s*\)', line)
        if m:
            utt_id = m.group(1)
            text = m.group(2)
            ref_map[utt_id] = normalize_text(text)
            continue

        if "|" in line:
            parts = line.split("|", 1)
            utt_id = parts[0].strip()
            text = parts[1].strip()
            ref_map[utt_id] = normalize_text(text)
            continue

        if "\t" in line:
            parts = line.split("\t", 1)
            utt_id = parts[0].strip()
            text = parts[1].strip()
            ref_map[utt_id] = normalize_text(text)
            continue

    if len(ref_map) == 0:
        raise RuntimeError("Failed to parse transcript file.")

    return ref_map

def collect_val_files_for_speaker(data_root, spk):
    mel_dir = os.path.join(data_root, "mel", spk)
    files = sorted([x for x in os.listdir(mel_dir) if x.endswith(".npy")])

    val_files = []
    for f in files:
        idx = parse_global_idx(f)
        if VAL_START <= idx <= VAL_END:
            val_files.append(f)

    return val_files

def resolve_ckpt(tag):
    if tag == "best":
        return os.path.join(CKPT_DIR, "best_model.pt"), "best_model"
    if tag == "latest":
        return os.path.join(CKPT_DIR, "latest_model.pt"), "latest_model"
    return os.path.join(CKPT_DIR, f"model_epoch_{tag}.pt"), f"ckpt_{tag}"

def build_ref_index(root):
    idx = {}
    for spk_dir in sorted(glob.glob(os.path.join(root, "*"))):
        spk = os.path.basename(spk_dir)
        if not os.path.isdir(spk_dir):
            continue

        files = glob.glob(os.path.join(spk_dir, "wav", "*.wav"))
        files += glob.glob(os.path.join(spk_dir, "*.wav"))
        if not files:
            files = glob.glob(os.path.join(spk_dir, "**", "*.wav"), recursive=True)

        for wav_path in files:
            utt_id = os.path.splitext(os.path.basename(wav_path))[0]
            idx[(spk, utt_id)] = wav_path
    return idx

def mean_std(arr):
    arr = pd.Series(arr).dropna().to_numpy(dtype=np.float64)
    if len(arr) == 0:
        return np.nan, np.nan
    return float(np.mean(arr)), float(np.std(arr, ddof=0))

def mean_ci95(arr):
    arr = pd.Series(arr).dropna().to_numpy(dtype=np.float64)
    n = len(arr)
    if n == 0:
        return np.nan, np.nan, 0
    mean = float(np.mean(arr))
    std = float(np.std(arr, ddof=0))
    ci95 = float(1.96 * std / np.sqrt(n))
    return mean, ci95, n

# 你已有的 test-set 汇总结果，直接写进来方便最后自动对比
TEST_SET_REFERENCE = pd.DataFrame([
    {"ckpt_name": "best_model", "test_cer_pct": 2.572, "test_mcd_db": 7.864, "test_lfc": 0.376, "test_pmos": 2.845},
    {"ckpt_name": "ckpt_500",  "test_cer_pct": 1.916, "test_mcd_db": 6.584, "test_lfc": 0.435, "test_pmos": 3.128},
    {"ckpt_name": "ckpt_1000", "test_cer_pct": 1.955, "test_mcd_db": 6.107, "test_lfc": 0.468, "test_pmos": 3.201},
    {"ckpt_name": "ckpt_1500", "test_cer_pct": 2.208, "test_mcd_db": 6.062, "test_lfc": 0.444, "test_pmos": 3.198},
    {"ckpt_name": "ckpt_2000", "test_cer_pct": 2.209, "test_mcd_db": 6.048, "test_lfc": 0.472, "test_pmos": 3.196},
    {"ckpt_name": "ckpt_2500", "test_cer_pct": 2.245, "test_mcd_db": 6.031, "test_lfc": 0.432, "test_pmos": 3.197},
    {"ckpt_name": "ckpt_3000", "test_cer_pct": 2.211, "test_mcd_db": 6.008, "test_lfc": 0.416, "test_pmos": 3.195},
    {"ckpt_name": "ckpt_3500", "test_cer_pct": 2.286, "test_mcd_db": 6.019, "test_lfc": 0.430, "test_pmos": 3.190},
    {"ckpt_name": "ckpt_4000", "test_cer_pct": 2.252, "test_mcd_db": 6.016, "test_lfc": 0.429, "test_pmos": 3.179},
])


In [ ]:

# ===== Cell 6: 检查数据与验证集列表 =====
required_dirs = [PROJECT_ROOT, DATA_ROOT, CKPT_DIR, HIFIGAN_ROOT]
required_files = [HIFIGAN_CONFIG, HIFIGAN_CKPT, TEXT_PATH]
for p in required_dirs:
    if not os.path.isdir(p):
        raise FileNotFoundError(f"目录不存在: {p}")
for p in required_files:
    if not os.path.exists(p):
        raise FileNotFoundError(f"文件不存在: {p}")

val_lists = {spk: collect_val_files_for_speaker(DATA_ROOT, spk) for spk in CLOSEDSET_SPKS}
for spk, items in val_lists.items():
    print(spk, len(items), items[:3], "...", items[-3:])
    if len(items) != 100:
        print(f"[Warning] {spk} validation count = {len(items)} (expected 100)")

expected_total = len(PAIR_ORDER) * 100
print("Expected converted wavs per checkpoint =", expected_total)

mel_mean, mel_std = load_stats(DATA_ROOT)
ref_map = load_ref_map(TEXT_PATH)
ref_index = build_ref_index(os.path.join(DATA_ROOT, "wav"))

print("Loaded refs:", len(ref_map))
print("Reference wav index:", len(ref_index))


In [ ]:

# ===== Cell 7: 加载 HiFi-GAN / ASR =====
if HIFIGAN_ROOT not in sys.path:
    sys.path.append(HIFIGAN_ROOT)

from env import AttrDict
from models import Generator

with open(HIFIGAN_CONFIG, "r") as f:
    h = AttrDict(json.load(f))

generator = Generator(h).to(DEVICE)
hifigan_ckpt = torch.load(HIFIGAN_CKPT, map_location=DEVICE)
if isinstance(hifigan_ckpt, dict) and "generator" in hifigan_ckpt:
    generator.load_state_dict(hifigan_ckpt["generator"])
else:
    generator.load_state_dict(hifigan_ckpt)
generator.eval()
generator.remove_weight_norm()

print("HiFi-GAN loaded OK")

if DO_CER:
    processor = Wav2Vec2Processor.from_pretrained(ASR_MODEL_NAME)
    asr_model = Wav2Vec2ForCTC.from_pretrained(ASR_MODEL_NAME).to(DEVICE)
    asr_model.eval()
    print("ASR loaded:", ASR_MODEL_NAME)
else:
    processor = None
    asr_model = None


In [ ]:

# ===== Cell 8: 模型与推理函数 =====
def build_model_and_diffusion(device=DEVICE):
    model = VoiceGrad(
        n_mels=80,
        n_bnf=144,
        n_channels=512,
        n_spk=4
    ).to(device)

    diffusion = VoiceGradDiffusion(
        n_levels=20,
        offset=0.008
    ).to(device)

    return model, diffusion

def load_ckpt_to_model(ckpt_path, model, map_location="cpu"):
    ckpt = torch.load(ckpt_path, map_location=map_location)

    if isinstance(ckpt, dict) and "model" in ckpt:
        state_dict = ckpt["model"]
        epoch = ckpt.get("epoch", "unknown")
    elif isinstance(ckpt, dict) and "state_dict" in ckpt:
        state_dict = ckpt["state_dict"]
        epoch = ckpt.get("epoch", "unknown")
    else:
        state_dict = ckpt
        epoch = "unknown"

    model.load_state_dict(state_dict, strict=True)
    return epoch

@torch.no_grad()
def convert_one(model, diffusion, src_spk, src_mel_file, tgt_spk):
    mel_np, bnf_np, _, _ = load_sample_from_data(DATA_ROOT, src_spk, src_mel_file)

    mel_norm = normalize_mel(mel_np, mel_mean, mel_std).to(DEVICE)
    bnf_tensor = torch.from_numpy(bnf_np).float().unsqueeze(0).to(DEVICE)
    target_id = torch.tensor([SPK2ID[tgt_spk]], dtype=torch.long, device=DEVICE)

    converted_norm = diffusion.sample(
        model=model,
        x_source=mel_norm,
        speaker_id=target_id,
        bnf=bnf_tensor,
        start_level=START_LEVEL
    )

    converted_mel = denormalize_mel(converted_norm.cpu(), mel_mean, mel_std)
    return converted_mel


In [ ]:

# ===== Cell 9: 指标函数 =====
@torch.no_grad()
def transcribe_wav_ctc(wav_path, target_sr=16000):
    wav, sr = sf.read(wav_path)
    if wav.ndim == 2:
        wav = wav.mean(axis=1)
    if sr != target_sr:
        wav = librosa.resample(wav.astype(np.float32), orig_sr=sr, target_sr=target_sr)
        sr = target_sr

    inputs = processor(
        wav,
        sampling_rate=sr,
        return_tensors="pt",
        padding=True
    )

    input_values = inputs.input_values.to(DEVICE)
    attention_mask = inputs.attention_mask.to(DEVICE) if "attention_mask" in inputs else None

    logits = asr_model(input_values, attention_mask=attention_mask).logits
    pred_ids = torch.argmax(logits, dim=-1)
    text = processor.batch_decode(pred_ids)[0]
    return normalize_text(text)

def load_wav_mono_16k(wav_path, sr=SR):
    wav, file_sr = sf.read(wav_path)
    if wav.ndim == 2:
        wav = wav.mean(axis=1)
    wav = wav.astype(np.float64)
    if file_sr != sr:
        wav = librosa.resample(
            wav.astype(np.float32),
            orig_sr=file_sr,
            target_sr=sr
        ).astype(np.float64)
    return wav

def extract_f0_mcep(wav_path):
    wav = load_wav_mono_16k(wav_path)

    f0, timeaxis = pw.harvest(
        wav,
        SR,
        frame_period=FRAME_PERIOD_MS,
        f0_floor=20.0,
        f0_ceil=600.0
    )
    sp = pw.cheaptrick(wav, f0, timeaxis, SR)
    mcep = pysptk.sp2mc(sp, order=MCEP_ORDER, alpha=MCEP_ALPHA)

    return {
        "f0": f0.astype(np.float64),
        "mcep": mcep.astype(np.float64),
    }

def score_one_mcd_lfc(gen_wav_path, ref_wav_path):
    g = extract_f0_mcep(gen_wav_path)
    r = extract_f0_mcep(ref_wav_path)

    C = cdist(g["mcep"][:, 1:], r["mcep"][:, 1:], metric="euclidean")
    _, wp = dtw(C=C)
    wp = np.array(wp[::-1], dtype=np.int64)

    dists = []
    xs, ys = [], []

    for i, j in wp:
        diff = g["mcep"][i, 1:] - r["mcep"][j, 1:]
        dists.append(MCD_CONST * np.sqrt(np.sum(diff * diff)))

        if g["f0"][i] > 0 and r["f0"][j] > 0:
            xs.append(np.log(g["f0"][i]))
            ys.append(np.log(r["f0"][j]))

    mcd = float(np.mean(dists))

    xs = np.asarray(xs)
    ys = np.asarray(ys)
    if len(xs) < 2 or np.std(xs) < 1e-12 or np.std(ys) < 1e-12:
        lfc = np.nan
    else:
        lfc = float(np.corrcoef(xs, ys)[0, 1])

    return mcd, lfc

def collect_generated_wavs(root):
    rows = []
    for pair_dir in sorted(glob.glob(os.path.join(root, "*_to_*"))):
        m = re.match(r"([a-z]+)_to_([a-z]+)", os.path.basename(pair_dir))
        if not m:
            continue
        src_spk, tgt_spk = m.group(1), m.group(2)

        for wav_path in sorted(glob.glob(os.path.join(pair_dir, "*.wav"))):
            utt_id = os.path.splitext(os.path.basename(wav_path))[0]
            rows.append({
                "src_spk": src_spk,
                "tgt_spk": tgt_spk,
                "utt_id": utt_id,
                "wav_path": wav_path,
            })
    return pd.DataFrame(rows)


In [ ]:

# ===== Cell 10: 单个 checkpoint 的完整评测流程 =====
def evaluate_one_checkpoint(tag):
    ckpt_path, ckpt_name = resolve_ckpt(tag)
    if not os.path.exists(ckpt_path):
        print(f"[Skip] checkpoint not found: {ckpt_path}")
        return None

    ckpt_out = os.path.join(OUT_BASE, ckpt_name)
    gen_root = os.path.join(ckpt_out, "generated_wavs")
    os.makedirs(gen_root, exist_ok=True)

    done_flag = os.path.join(ckpt_out, "all_done.json")
    if SKIP_IF_DONE and os.path.exists(done_flag):
        print(f"[Skip done] {ckpt_name}")
        with open(done_flag, "r", encoding="utf-8") as f:
            return json.load(f)

    print("=" * 100)
    print("Evaluating:", ckpt_name)
    print("ckpt path  :", ckpt_path)

    model, diffusion = build_model_and_diffusion(device=DEVICE)
    loaded_epoch = load_ckpt_to_model(ckpt_path, model, map_location=DEVICE)
    model.eval()
    print("loaded epoch =", loaded_epoch)

    # --------------------------------------------------
    # A. 生成 wav
    # --------------------------------------------------
    meta_rows = []
    for src_spk, tgt_spk in PAIR_ORDER:
        pair_dir = os.path.join(gen_root, f"{src_spk}_to_{tgt_spk}")
        os.makedirs(pair_dir, exist_ok=True)

        src_files = val_lists[src_spk]
        print(f"Running pair: {src_spk} -> {tgt_spk} | n={len(src_files)}")

        for src_mel_file in tqdm(src_files, leave=False):
            utt_id = src_mel_file.replace(".npy", "")
            wav_out_path = os.path.join(pair_dir, f"{utt_id}.wav")

            if not (REUSE_EXISTING_WAVS and os.path.exists(wav_out_path)):
                converted_mel = convert_one(model, diffusion, src_spk, src_mel_file, tgt_spk)
                y_g_hat = generator(converted_mel.to(DEVICE))
                wav = y_g_hat.squeeze().detach().cpu().numpy()
                sf.write(wav_out_path, wav, h.sampling_rate)

            meta_rows.append({
                "src_spk": src_spk,
                "tgt_spk": tgt_spk,
                "utt_id": utt_id,
                "wav_path": wav_out_path,
            })

    meta_df = pd.DataFrame(meta_rows)
    meta_csv = os.path.join(ckpt_out, "generated_meta.csv")
    meta_df.to_csv(meta_csv, index=False, encoding="utf-8-sig")

    print("Generated wav count =", len(meta_df), "expected =", len(PAIR_ORDER) * 100)

    # 释放模型显存
    del model, diffusion
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # --------------------------------------------------
    # B. CER
    # --------------------------------------------------
    cer_detail_csv = os.path.join(ckpt_out, "cer_detail.csv")
    if DO_CER:
        if REUSE_EXISTING_METRICS and os.path.exists(cer_detail_csv):
            cer_df = pd.read_csv(cer_detail_csv)
        else:
            rows = []
            for item in tqdm(meta_rows, desc=f"CER {ckpt_name}"):
                utt_id = item["utt_id"]
                wav_path = item["wav_path"]

                if utt_id not in ref_map:
                    continue

                ref = ref_map[utt_id]
                hyp = transcribe_wav_ctc(wav_path)
                cur_cer = cer(ref, hyp)

                rows.append({
                    "src_spk": item["src_spk"],
                    "tgt_spk": item["tgt_spk"],
                    "utt_id": utt_id,
                    "ref": ref,
                    "hyp": hyp,
                    "cer": cur_cer,
                    "wav_path": wav_path
                })

            cer_df = pd.DataFrame(rows)
            cer_df.to_csv(cer_detail_csv, index=False, encoding="utf-8-sig")
    else:
        cer_df = pd.DataFrame()

    # --------------------------------------------------
    # C. MCD / LFC
    # --------------------------------------------------
    mcd_lfc_detail_csv = os.path.join(ckpt_out, "mcd_lfc_detail.csv")
    if DO_MCD_LFC:
        if REUSE_EXISTING_METRICS and os.path.exists(mcd_lfc_detail_csv):
            mcd_lfc_df = pd.read_csv(mcd_lfc_detail_csv)
        else:
            rows = []
            for item in tqdm(meta_rows, desc=f"MCD/LFC {ckpt_name}"):
                ref_wav_path = ref_index.get((item["tgt_spk"], item["utt_id"]), None)
                if ref_wav_path is None:
                    rows.append({
                        **item,
                        "ref_wav_path": None,
                        "mcd_db": np.nan,
                        "lfc": np.nan,
                    })
                    continue

                try:
                    mcd, lfc = score_one_mcd_lfc(item["wav_path"], ref_wav_path)
                except Exception as e:
                    print("[MCD/LFC error]", item["wav_path"], "=>", e)
                    mcd, lfc = np.nan, np.nan

                rows.append({
                    **item,
                    "ref_wav_path": ref_wav_path,
                    "mcd_db": mcd,
                    "lfc": lfc,
                })

            mcd_lfc_df = pd.DataFrame(rows)
            mcd_lfc_df.to_csv(mcd_lfc_detail_csv, index=False, encoding="utf-8-sig")
    else:
        mcd_lfc_df = pd.DataFrame()

    # --------------------------------------------------
    # D. pMOS (DNSMOS)
    # --------------------------------------------------
    pmos_detail_csv = os.path.join(ckpt_out, "pmos_dnsmos_detail.csv")
    if DO_PMOS:
        if REUSE_EXISTING_METRICS and os.path.exists(pmos_detail_csv):
            pmos_df = pd.read_csv(pmos_detail_csv)
        else:
            rows = []
            for item in tqdm(meta_rows, desc=f"pMOS {ckpt_name}"):
                wav_path = item["wav_path"]
                try:
                    audio, sr = librosa.load(wav_path, sr=16000)
                    scores = dnsmos.run(audio, sr)
                    pmos_score = float(scores["ovrl_mos"])
                except Exception as e:
                    print("[DNSMOS error]", wav_path, "=>", e)
                    pmos_score = np.nan

                rows.append({
                    "src_spk": item["src_spk"],
                    "tgt_spk": item["tgt_spk"],
                    "utt_id": item["utt_id"],
                    "wav_path": wav_path,
                    "pmos": pmos_score,
                })

            pmos_df = pd.DataFrame(rows)
            pmos_df.to_csv(pmos_detail_csv, index=False, encoding="utf-8-sig")
    else:
        pmos_df = pd.DataFrame()

    # --------------------------------------------------
    # E. 汇总
    # --------------------------------------------------
    summary_rows = []
    for src_spk, tgt_spk in PAIR_ORDER:
        row = {
            "src_spk": src_spk,
            "tgt_spk": tgt_spk,
            "count": 100,
        }

        if DO_CER and len(cer_df) > 0:
            sub = cer_df[(cer_df["src_spk"] == src_spk) & (cer_df["tgt_spk"] == tgt_spk)]
            row["cer_mean_pct"] = float(sub["cer"].mean() * 100.0) if len(sub) > 0 else np.nan

        if DO_MCD_LFC and len(mcd_lfc_df) > 0:
            sub = mcd_lfc_df[(mcd_lfc_df["src_spk"] == src_spk) & (mcd_lfc_df["tgt_spk"] == tgt_spk)]
            row["mcd_mean_db"] = float(sub["mcd_db"].mean()) if len(sub) > 0 else np.nan
            row["lfc_mean"] = float(sub["lfc"].mean()) if len(sub) > 0 else np.nan

        if DO_PMOS and len(pmos_df) > 0:
            sub = pmos_df[(pmos_df["src_spk"] == src_spk) & (pmos_df["tgt_spk"] == tgt_spk)]
            row["pmos_mean"] = float(sub["pmos"].mean()) if len(sub) > 0 else np.nan

        summary_rows.append(row)

    pair_summary = pd.DataFrame(summary_rows)
    pair_summary_csv = os.path.join(ckpt_out, "summary_by_pair.csv")
    pair_summary.to_csv(pair_summary_csv, index=False, encoding="utf-8-sig")

    all_result = {
        "ckpt_name": ckpt_name,
        "ckpt_path": ckpt_path,
        "loaded_epoch": str(loaded_epoch),
        "n_generated": int(len(meta_df)),
        "val_items_per_pair": 100,
        "val_total_pairs": int(len(PAIR_ORDER)),
        "val_total_items": int(len(PAIR_ORDER) * 100),
        "cer_pct": float(cer_df["cer"].mean() * 100.0) if DO_CER and len(cer_df) > 0 else np.nan,
        "mcd_db": float(mcd_lfc_df["mcd_db"].mean()) if DO_MCD_LFC and len(mcd_lfc_df) > 0 else np.nan,
        "lfc": float(mcd_lfc_df["lfc"].mean()) if DO_MCD_LFC and len(mcd_lfc_df) > 0 else np.nan,
        "pmos": float(pmos_df["pmos"].mean()) if DO_PMOS and len(pmos_df) > 0 else np.nan,
        "meta_csv": meta_csv,
        "pair_summary_csv": pair_summary_csv,
        "cer_detail_csv": cer_detail_csv if DO_CER else None,
        "mcd_lfc_detail_csv": mcd_lfc_detail_csv if DO_MCD_LFC else None,
        "pmos_detail_csv": pmos_detail_csv if DO_PMOS else None,
    }

    with open(done_flag, "w", encoding="utf-8") as f:
        json.dump(all_result, f, ensure_ascii=False, indent=2)

    print("Done:", ckpt_name)
    print(json.dumps(all_result, ensure_ascii=False, indent=2))
    return all_result


In [ ]:

# ===== Cell 11: 跑全部 checkpoint =====
results = []

for tag in EVAL_CKPTS:
    try:
        out = evaluate_one_checkpoint(tag)
        if out is not None:
            results.append(out)
    except Exception as e:
        print("=" * 100)
        print("[FATAL ERROR]", tag, "=>", e)
        print("=" * 100)

summary_df = pd.DataFrame(results)

# 固定顺序
order_map = {
    "best_model": 0,
    "ckpt_500": 1,
    "ckpt_1000": 2,
    "ckpt_1500": 3,
    "ckpt_2000": 4,
    "ckpt_2500": 5,
    "ckpt_3000": 6,
    "ckpt_3500": 7,
    "ckpt_4000": 8,
    "latest_model": 9,
}
if len(summary_df) > 0:
    summary_df["__order"] = summary_df["ckpt_name"].map(order_map)
    summary_df = summary_df.sort_values("__order").drop(columns="__order").reset_index(drop=True)

summary_csv = os.path.join(OUT_BASE, "val_checkpoint_summary.csv")
summary_df.to_csv(summary_csv, index=False, encoding="utf-8-sig")

print("Saved:", summary_csv)
display(summary_df)


In [ ]:

# ===== Cell 12: 和已有 test-set 结果自动对比 =====
if os.path.exists(os.path.join(OUT_BASE, "val_checkpoint_summary.csv")):
    val_df = pd.read_csv(os.path.join(OUT_BASE, "val_checkpoint_summary.csv"))
else:
    val_df = pd.DataFrame()

if len(val_df) == 0:
    print("No validation summary found.")
else:
    compare_df = val_df.merge(TEST_SET_REFERENCE, on="ckpt_name", how="left")

    compare_df["delta_cer_val_minus_test"] = compare_df["cer_pct"] - compare_df["test_cer_pct"]
    compare_df["delta_mcd_val_minus_test"] = compare_df["mcd_db"] - compare_df["test_mcd_db"]
    compare_df["delta_lfc_val_minus_test"] = compare_df["lfc"] - compare_df["test_lfc"]
    compare_df["delta_pmos_val_minus_test"] = compare_df["pmos"] - compare_df["test_pmos"]

    compare_csv = os.path.join(OUT_BASE, "val_vs_test_compare.csv")
    compare_df.to_csv(compare_csv, index=False, encoding="utf-8-sig")

    print("Saved:", compare_csv)
    display(compare_df[[
        "ckpt_name",
        "cer_pct", "test_cer_pct", "delta_cer_val_minus_test",
        "mcd_db", "test_mcd_db", "delta_mcd_val_minus_test",
        "lfc", "test_lfc", "delta_lfc_val_minus_test",
        "pmos", "test_pmos", "delta_pmos_val_minus_test",
    ]])
